In [2]:
import re
import pandas as pd


def parse_can_log(path):
    """
    Parse CAN log lines like:
    (1290000000.012715) can0 230#FD000002D4000400
    Returns a DataFrame with columns: Timestamp (float), ID (hex str), Data (hex str), ID_int (int)
    """
    pattern = re.compile(r'^\((?P<ts>\d+\.\d+)\)\s+\S+\s+(?P<id>[0-9A-Fa-f]+)#(?P<data>[0-9A-Fa-f]+)')
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            m = pattern.match(line.strip())
            if not m:
                continue
            ts = float(m.group('ts'))
            id_hex = m.group('id').upper()
            data_hex = m.group('data').upper()
            try:
                id_int = int(id_hex, 16)
            except ValueError:
                id_int = None
            rows.append((ts, id_hex, data_hex, id_int))
    return pd.DataFrame(rows, columns=['Timestamp', 'ID', 'Data', 'ID_int'])

In [3]:
import os
import json
import pandas as pd

# Load metadata
with open(r"C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\datasets\road_dataset\road\attacks\capture_metadata.json", "r") as f:
    metadata = json.load(f)

def load_attack(log_name):
    """
    Load a ROAD attack log and label injected packets.
    """

    path = os.path.join(
        r"C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks",
        "datasets",
        "road_dataset",
        "road",
        "attacks",
        log_name + ".log"
    )

    # Load log
    df = parse_can_log(path)

    # Use the first timestamp in the capture as the base
    base_timestamp = df["Timestamp"].iloc[0]

    # Default label
    df["Attack"] = "R"

    # Look up attack interval
    info = metadata.get(log_name)

    if info is not None and info["injection_interval"] is not None:

        start, end = info["injection_interval"]

        df.loc[
            (df["Timestamp"] >= base_timestamp + start) &
            (df["Timestamp"] <= base_timestamp + end),
            "Attack"
        ] = "T"

    return df



In [4]:
import os

def load_ambient(log_name):
    """
    Load a ROAD ambient log.
    All packets are labelled as Regular ('R').
    """

    path = os.path.join(
        r"C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks",
        "datasets",
        "road_dataset",
        "road",
        "ambient",
        log_name + ".log"
    )

    # Load log
    df = parse_can_log(path)

    # All packets are normal
    df["Attack"] = "R"

    return df

In [5]:
AMBIENT_FOLDER = (
    r"C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks"
    r"\datasets\road_dataset\road\ambient"
)

ambient_dfs = {}

for filename in os.listdir(AMBIENT_FOLDER):

    if not filename.endswith(".log"):
        continue

    log_name = filename[:-4]  # Remove ".log"

    print(f"Loading {log_name}...")

    ambient_dfs[log_name] = load_ambient(log_name)

print(f"\nLoaded {len(ambient_dfs)} ambient datasets.")

Loading ambient_dyno_drive_basic_long...
Loading ambient_dyno_drive_basic_short...
Loading ambient_dyno_drive_benign_anomaly...
Loading ambient_dyno_drive_extended_long...
Loading ambient_dyno_drive_extended_short...
Loading ambient_dyno_drive_radio_infotainment...
Loading ambient_dyno_drive_winter...
Loading ambient_dyno_exercise_all_bits...
Loading ambient_dyno_idle_radio_infotainment...
Loading ambient_dyno_reverse...
Loading ambient_highway_street_driving_diagnostics...
Loading ambient_highway_street_driving_long...

Loaded 12 ambient datasets.


In [6]:
import os

ATTACK_FOLDER = (
    r"C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks"
    r"\datasets\road_dataset\road\attacks"
)

# Dictionary of all attack DataFrames
attack_dfs = {}

for filename in os.listdir(ATTACK_FOLDER):

    # Only load .log files
    if not filename.endswith(".log"):
        continue

    attack_name = filename[:-4]  # Remove ".log"

    print(f"Loading {attack_name}...")

    attack_dfs[attack_name] = load_attack(attack_name)

print(f"\nLoaded {len(attack_dfs)} attack datasets.")

Loading accelerator_attack_drive_1...
Loading accelerator_attack_drive_2...
Loading accelerator_attack_reverse_1...
Loading accelerator_attack_reverse_2...
Loading correlated_signal_attack_1...
Loading correlated_signal_attack_1_masquerade...
Loading correlated_signal_attack_2...
Loading correlated_signal_attack_2_masquerade...
Loading correlated_signal_attack_3...
Loading correlated_signal_attack_3_masquerade...
Loading fuzzing_attack_1...
Loading fuzzing_attack_2...
Loading fuzzing_attack_3...
Loading max_engine_coolant_temp_attack...
Loading max_engine_coolant_temp_attack_masquerade...
Loading max_speedometer_attack_1...
Loading max_speedometer_attack_1_masquerade...
Loading max_speedometer_attack_2...
Loading max_speedometer_attack_2_masquerade...
Loading max_speedometer_attack_3...
Loading max_speedometer_attack_3_masquerade...
Loading reverse_light_off_attack_1...
Loading reverse_light_off_attack_1_masquerade...
Loading reverse_light_off_attack_2...
Loading reverse_light_off_atta

In [7]:
attack_dfs

{'accelerator_attack_drive_1':            Timestamp   ID              Data  ID_int Attack
 0       1.290000e+09  585  0040000000020018    1413      R
 1       1.290000e+09  354  1FFF40000003C580     852      R
 2       1.290000e+09  5E1  893FE0070A000080    1505      R
 3       1.290000e+09  28B  0000000000000000     651      R
 4       1.290000e+09  0A7  005108E5112A00A0     167      R
 ...              ...  ...               ...     ...    ...
 204755  1.290000e+09  153  00000000000C1002     339      R
 204756  1.290000e+09  662  4E60000040000000    1634      R
 204757  1.290000e+09  19C  02FC200002002730     412      R
 204758  1.290000e+09  00E  2054560208087530      14      R
 204759  1.290000e+09  0D0  0273046000000000     208      R
 
 [204760 rows x 5 columns],
 'accelerator_attack_drive_2':            Timestamp   ID              Data  ID_int Attack
 0       1.300000e+09  498  87FFBFFCF9800003    1176      R
 1       1.300000e+09  464  000240000090145A    1124      R
 2       1

In [8]:
import numpy as np

def sliding_windows_id_data(df, window_size=64, step=2, attack_label='T'):
    ids = df['ID'].to_numpy(dtype=object)
    data = df['Data'].to_numpy(dtype=object)
    attack = df['Attack'].to_numpy(dtype=object)

    n = len(df)
    num_windows = (n - window_size) // step + 1
    windows = []
    window_labels = []

    for start in range(0, n - window_size + 1, step):
        end = start + window_size
        windows.append(np.column_stack((ids[start:end], data[start:end])))
        window_labels.append(attack_label if 'T' in attack[start:end] else 'R')

    return windows, np.array(window_labels, dtype=object)


In [9]:
def _hex_byte_list_from_row(row):
    if isinstance(row, np.ndarray):
        row = row.tolist()

    if isinstance(row, (list, tuple)):
        if len(row) == 2:
            msg_id, data = row
        elif len(row) == 0:
            raise ValueError("Encountered an empty row")
        else:
            msg_id, data = row[0], row[1]
    else:
        raise ValueError(f"Expected a row-like value, got: {row}")

    msg_id = str(msg_id).strip()
    if len(msg_id) % 2 != 0:
        msg_id = '0' + msg_id

    id_bytes = [int(msg_id[i:i+2], 16) for i in range(0, len(msg_id), 2)]
    data_bytes = []
    for item in str(data).split():
        if not item:
            continue
        try:
            data_bytes.append(int(item, 16))
        except ValueError:
            data_bytes.append(int(item))

    return id_bytes + data_bytes


def convert_windowed_id_data_to_ints(windowed_data, pad_value=0):
    int_windows = []
    for window in windowed_data:
        if isinstance(window, np.ndarray):
            window = window.tolist()

        rows = []
        for row in window:
            rows.append(_hex_byte_list_from_row(row))

        max_len = max(len(r) for r in rows)
        padded_rows = [r + [pad_value] * (max_len - len(r)) for r in rows]
        int_windows.append(padded_rows)
    return int_windows

def format_id_data_windows(windows):
    formatted_all = []
    for window in windows:
        window = np.asarray(window, dtype=object)
        formatted = []
        for msg_id, data in window:
            id_str = str(msg_id).strip().lower().zfill(4)
            data_str = str(data).strip().lower()
            data_str = re.sub(r'[^0-9a-f]', '', data_str)
            if len(data_str) % 2:
                data_str = '0' + data_str
            data_bytes = [data_str[i:i+2] for i in range(0, len(data_str), 2)] if data_str else []
            formatted.append([id_str, ' '.join(data_bytes)])
        formatted_all.append(np.array(formatted, dtype=object))
    return np.array(formatted_all, dtype=object)

In [10]:
# Store all processed versions of every attack
windowed_datasets = {}

for attack_name, df in attack_dfs.items():

    # Create sliding windows
    windows, labels = sliding_windows_id_data(
        df,
        window_size=64,
        step=2,
        attack_label='T'
    )

    # Format ID/Data strings
    formatted_windows = format_id_data_windows(windows)

    # Convert to integer arrays
    int_windows = convert_windowed_id_data_to_ints(formatted_windows)
    int_windows = np.array(int_windows)

    # One-hot encode labels
    # R -> [1, 0]
    # T -> [0, 1]
    onehot_labels = np.array([
        [1, 0] if label == 'R' else [0, 1]
        for label in labels
    ], dtype=np.uint8)

    # Store everything
    windowed_datasets[attack_name] = {
        #"windows": windows,
        #"formatted_windows": formatted_windows,
        "int_windows": int_windows,
        #"labels": labels,
        "onehot_labels": onehot_labels
    }

print(f"Processed {len(windowed_datasets)} datasets.")

Processed 33 datasets.


In [11]:


for name, df in ambient_dfs.items():

    # Create sliding windows
    windows, labels = sliding_windows_id_data(
        df[:204760*2],
        window_size=64,
        step=2,
        attack_label='T'
    )

    # Format ID/Data strings
    formatted_windows = format_id_data_windows(windows)

    # Convert to integer arrays
    int_windows = convert_windowed_id_data_to_ints(formatted_windows)
    int_windows = np.array(int_windows)

    # One-hot encode labels
    # R -> [1, 0]
    # T -> [0, 1]
    onehot_labels = np.array([
        [1, 0] if label == 'R' else [0, 1]
        for label in labels
    ], dtype=np.uint8)

    # Store everything
    windowed_datasets[name] = {
        #"windows": windows,
        #"formatted_windows": formatted_windows,
        "int_windows": int_windows,
        #"labels": labels,
        "onehot_labels": onehot_labels
    }
#
print(f"Processed {len(windowed_datasets)} datasets.")

Processed 45 datasets.


In [ ]:
#!pip install joblib

In [ ]:
#save the windowed datasets to a pickle file
from joblib import dump

dump(windowed_datasets, "windowed_datasets.joblib", compress=3)

KeyboardInterrupt: 

In [ ]:
#read the windowed datasets from the pickle file
from joblib import load

windowed_datasets = load("windowed_datasets.joblib")

In [14]:
windowed_datasets["fuzzing_attack_2"]["int_windows"].shape

(16144, 64, 10)

In [18]:
windowed_datasets["fuzzing_attack_2"]["onehot_labels"]

array([[1, 0],
       [1, 0],
       [1, 0],
       ...,
       [0, 1],
       [0, 1],
       [0, 1]], shape=(16144, 2), dtype=uint8)

In [11]:
t_indices = np.nonzero(windowed_datasets["fuzzing_attack_3"]["labels"] == 'T')[0]
print("num T labels:", len(t_indices))
print("first T index:", t_indices[0] if len(t_indices) else None)
t_indices[:20]

num T labels: 848
first T index: 5740


array([5740, 5741, 5742, 5743, 5744, 5745, 5746, 5747, 5748, 5749, 5750,
       5751, 5752, 5753, 5754, 5755, 5756, 5757, 5758, 5759])

In [21]:
X = []

# Accelerator attacks (no third capture)
X.extend(windowed_datasets["accelerator_attack_drive_1"]["int_windows"])
X.extend(windowed_datasets["accelerator_attack_drive_2"]["int_windows"])
X.extend(windowed_datasets["accelerator_attack_reverse_1"]["int_windows"])
X.extend(windowed_datasets["accelerator_attack_reverse_2"]["int_windows"])

# Correlated signal attacks
X.extend(windowed_datasets["correlated_signal_attack_1"]["int_windows"])
X.extend(windowed_datasets["correlated_signal_attack_1_masquerade"]["int_windows"])
X.extend(windowed_datasets["correlated_signal_attack_2"]["int_windows"])
X.extend(windowed_datasets["correlated_signal_attack_2_masquerade"]["int_windows"])

# Fuzzing attacks
X.extend(windowed_datasets["fuzzing_attack_1"]["int_windows"])
X.extend(windowed_datasets["fuzzing_attack_2"]["int_windows"])

# Engine coolant attacks
X.extend(windowed_datasets["max_engine_coolant_temp_attack"]["int_windows"])
X.extend(windowed_datasets["max_engine_coolant_temp_attack_masquerade"]["int_windows"])

# Speedometer attacks
X.extend(windowed_datasets["max_speedometer_attack_1"]["int_windows"])
X.extend(windowed_datasets["max_speedometer_attack_1_masquerade"]["int_windows"])
X.extend(windowed_datasets["max_speedometer_attack_2"]["int_windows"])
X.extend(windowed_datasets["max_speedometer_attack_2_masquerade"]["int_windows"])

# Reverse light OFF attacks
X.extend(windowed_datasets["reverse_light_off_attack_1"]["int_windows"])
X.extend(windowed_datasets["reverse_light_off_attack_1_masquerade"]["int_windows"])
X.extend(windowed_datasets["reverse_light_off_attack_2"]["int_windows"])
X.extend(windowed_datasets["reverse_light_off_attack_2_masquerade"]["int_windows"])

# Reverse light ON attacks
X.extend(windowed_datasets["reverse_light_on_attack_1"]["int_windows"])
X.extend(windowed_datasets["reverse_light_on_attack_1_masquerade"]["int_windows"])
X.extend(windowed_datasets["reverse_light_on_attack_2"]["int_windows"])
X.extend(windowed_datasets["reverse_light_on_attack_2_masquerade"]["int_windows"])

#ambient
X.extend(windowed_datasets["ambient_dyno_drive_basic_long"]["int_windows"])
X.extend(windowed_datasets["ambient_dyno_drive_basic_short"]["int_windows"])
X.extend(windowed_datasets["ambient_dyno_drive_benign_anomaly"]["int_windows"])
X.extend(windowed_datasets["ambient_dyno_drive_extended_long"]["int_windows"])
X.extend(windowed_datasets["ambient_dyno_drive_extended_short"]["int_windows"])
X.extend(windowed_datasets["ambient_dyno_drive_radio_infotainment"]["int_windows"])
X.extend(windowed_datasets["ambient_dyno_drive_winter"]["int_windows"])
X.extend(windowed_datasets["ambient_dyno_exercise_all_bits"]["int_windows"])

"""
Loading ambient_dyno_drive_basic_long...
Loading ambient_dyno_drive_basic_short...
Loading ambient_dyno_drive_benign_anomaly...
Loading ambient_dyno_drive_extended_long...
Loading ambient_dyno_drive_extended_short...
Loading ambient_dyno_drive_radio_infotainment...
Loading ambient_dyno_drive_winter...
Loading ambient_dyno_exercise_all_bits...
Loading ambient_dyno_idle_radio_infotainment...
Loading ambient_dyno_reverse...
Loading ambient_highway_street_driving_diagnostics...
Loading ambient_highway_street_driving_long...
"""

# ---------------------------------------------------------
# Test set (all third captures)
# ---------------------------------------------------------

X_test = []

# Correlated signal attacks
X_test.extend(windowed_datasets["correlated_signal_attack_3"]["int_windows"])
X_test.extend(windowed_datasets["correlated_signal_attack_3_masquerade"]["int_windows"])

# Fuzzing attacks
X_test.extend(windowed_datasets["fuzzing_attack_3"]["int_windows"])

# Speedometer attacks
X_test.extend(windowed_datasets["max_speedometer_attack_3"]["int_windows"])
X_test.extend(windowed_datasets["max_speedometer_attack_3_masquerade"]["int_windows"])

# Reverse light OFF attacks
X_test.extend(windowed_datasets["reverse_light_off_attack_3"]["int_windows"])
X_test.extend(windowed_datasets["reverse_light_off_attack_3_masquerade"]["int_windows"])

# Reverse light ON attacks
X_test.extend(windowed_datasets["reverse_light_on_attack_3"]["int_windows"])
X_test.extend(windowed_datasets["reverse_light_on_attack_3_masquerade"]["int_windows"])

#ambient
X_test.extend(windowed_datasets["ambient_dyno_idle_radio_infotainment"]["int_windows"])
X_test.extend(windowed_datasets["ambient_dyno_reverse"]["int_windows"])
X_test.extend(windowed_datasets["ambient_highway_street_driving_diagnostics"]["int_windows"])
X_test.extend(windowed_datasets["ambient_highway_street_driving_long"]["int_windows"])



In [22]:
y = []

# Accelerator attacks (no third capture)
y.extend(windowed_datasets["accelerator_attack_drive_1"]["onehot_labels"])
y.extend(windowed_datasets["accelerator_attack_drive_2"]["onehot_labels"])
y.extend(windowed_datasets["accelerator_attack_reverse_1"]["onehot_labels"])
y.extend(windowed_datasets["accelerator_attack_reverse_2"]["onehot_labels"])

# Correlated signal attacks
y.extend(windowed_datasets["correlated_signal_attack_1"]["onehot_labels"])
y.extend(windowed_datasets["correlated_signal_attack_1_masquerade"]["onehot_labels"])
y.extend(windowed_datasets["correlated_signal_attack_2"]["onehot_labels"])
y.extend(windowed_datasets["correlated_signal_attack_2_masquerade"]["onehot_labels"])

# Fuzzing attacks
y.extend(windowed_datasets["fuzzing_attack_1"]["onehot_labels"])
y.extend(windowed_datasets["fuzzing_attack_2"]["onehot_labels"])

# Engine coolant attacks
y.extend(windowed_datasets["max_engine_coolant_temp_attack"]["onehot_labels"])
y.extend(windowed_datasets["max_engine_coolant_temp_attack_masquerade"]["onehot_labels"])

# Speedometer attacks
y.extend(windowed_datasets["max_speedometer_attack_1"]["onehot_labels"])
y.extend(windowed_datasets["max_speedometer_attack_1_masquerade"]["onehot_labels"])
y.extend(windowed_datasets["max_speedometer_attack_2"]["onehot_labels"])
y.extend(windowed_datasets["max_speedometer_attack_2_masquerade"]["onehot_labels"])

# Reverse light OFF attacks
y.extend(windowed_datasets["reverse_light_off_attack_1"]["onehot_labels"])
y.extend(windowed_datasets["reverse_light_off_attack_1_masquerade"]["onehot_labels"])
y.extend(windowed_datasets["reverse_light_off_attack_2"]["onehot_labels"])
y.extend(windowed_datasets["reverse_light_off_attack_2_masquerade"]["onehot_labels"])

# Reverse light ON attacks
y.extend(windowed_datasets["reverse_light_on_attack_1"]["onehot_labels"])
y.extend(windowed_datasets["reverse_light_on_attack_1_masquerade"]["onehot_labels"])
y.extend(windowed_datasets["reverse_light_on_attack_2"]["onehot_labels"])
y.extend(windowed_datasets["reverse_light_on_attack_2_masquerade"]["onehot_labels"])

#ambient
y.extend(windowed_datasets["ambient_dyno_drive_basic_long"]["onehot_labels"])
y.extend(windowed_datasets["ambient_dyno_drive_basic_short"]["onehot_labels"])
y.extend(windowed_datasets["ambient_dyno_drive_benign_anomaly"]["onehot_labels"])
y.extend(windowed_datasets["ambient_dyno_drive_extended_long"]["onehot_labels"])
y.extend(windowed_datasets["ambient_dyno_drive_extended_short"]["onehot_labels"])
y.extend(windowed_datasets["ambient_dyno_drive_radio_infotainment"]["onehot_labels"])
y.extend(windowed_datasets["ambient_dyno_drive_winter"]["onehot_labels"])
y.extend(windowed_datasets["ambient_dyno_exercise_all_bits"]["onehot_labels"])


# ---------------------------------------------------------
# Test labels (all third captures)
# ---------------------------------------------------------

y_test = []

# Correlated signal attacks
y_test.extend(windowed_datasets["correlated_signal_attack_3"]["onehot_labels"])
y_test.extend(windowed_datasets["correlated_signal_attack_3_masquerade"]["onehot_labels"])

# Fuzzing attacks
y_test.extend(windowed_datasets["fuzzing_attack_3"]["onehot_labels"])

# Speedometer attacks
y_test.extend(windowed_datasets["max_speedometer_attack_3"]["onehot_labels"])
y_test.extend(windowed_datasets["max_speedometer_attack_3_masquerade"]["onehot_labels"])

# Reverse light OFF attacks
y_test.extend(windowed_datasets["reverse_light_off_attack_3"]["onehot_labels"])
y_test.extend(windowed_datasets["reverse_light_off_attack_3_masquerade"]["onehot_labels"])

# Reverse light ON attacks
y_test.extend(windowed_datasets["reverse_light_on_attack_3"]["onehot_labels"])
y_test.extend(windowed_datasets["reverse_light_on_attack_3_masquerade"]["onehot_labels"])

#ambient
y_test.extend(windowed_datasets["ambient_dyno_idle_radio_infotainment"]["onehot_labels"])
y_test.extend(windowed_datasets["ambient_dyno_reverse"]["onehot_labels"])
y_test.extend(windowed_datasets["ambient_highway_street_driving_diagnostics"]["onehot_labels"])
y_test.extend(windowed_datasets["ambient_highway_street_driving_long"]["onehot_labels"])

In [23]:
import numpy as np

X = np.array(X, dtype=np.uint8)
X_test = np.array(X_test, dtype=np.uint8)

y = np.array(y, dtype=np.uint8)
y_test = np.array(y_test, dtype=np.uint8)

print(X.shape, y.shape)
print(X_test.shape, y_test.shape)

(2984078, 64, 10) (2984078, 2)
(1228225, 64, 10) (1228225, 2)


In [ ]:
from joblib import dump

dump(X, "saved_data/X.joblib", compress=3)
dump(y, "saved_data/y.joblib", compress=3)
dump(X_test, "saved_data/X_test.joblib", compress=3)
dump(y_test, "saved_data/y_test.joblib", compress=3)

['y_test.joblib']

In [32]:
from joblib import load

X = load("saved_data/X.joblib")
y = load("saved_data/y.joblib")
X_test = load("saved_data/X_test.joblib")
y_test = load("saved_data/y_test.joblib")

In [34]:
import tensorflow as tf
import tensorflow as tf

from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 10, 1)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(2, activation='softmax'))

In [35]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

mod = model.fit(X, y, epochs=5, 
          batch_size=2000, 
          validation_split=0.2)
          
print(mod)

Epoch 1/5
 256/1194 ━━━━━━━━━━━━━━━━━━━━ 2:06 135ms/step - accuracy: 0.8502 - loss: 1.3782

KeyboardInterrupt: 

In [26]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Convert one-hot labels to integer labels
y_true = np.argmax(y_test, axis=1)

# Predict probabilities and convert to integer labels
y_pred_probs = model.predict(X_test, batch_size=2000)
y_pred = np.argmax(y_pred_probs, axis=1)

# Compute metrics
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, pos_label=1)
rec = recall_score(y_true, y_pred, pos_label=1)
f1 = f1_score(y_true, y_pred, pos_label=1)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn) if (fp + tn) else 0.0
fnr = fn / (fn + tp) if (fn + tp) else 0.0

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 score:", f1)
print("Confusion matrix:\n", cm)
print("False positive rate:", fpr)
print("False negative rate:", fnr)

615/615 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step


c:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Accuracy: 0.7575419813144986
Precision: 0.0
Recall: 0.0
F1 score: 0.0
Confusion matrix:
 [[930432      0]
 [297793      0]]
False positive rate: 0.0
False negative rate: 1.0
